In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [2]:
from comet import download_model, load_from_checkpoint
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
import sacrebleu
from typing import List
from itertools import chain

/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-11-03 22:21:15.759176: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-03 22:21:15.811520: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-03 22:21:24.282525: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could n

In [3]:
checkpoint_path = download_model("Unbabel/wmt22-comet-da")
comet_metric = load_from_checkpoint(checkpoint_path)


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 8916.46it/s]


Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.2.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../scratch-artemis/sweta/cache/models--Unbabel--wmt22-comet-da/snapshots/371e9839ca4e213dde891b066cf3080f75ec7e72/checkpoints/model.ckpt`
Encoder model frozen.


In [4]:
def get_candidates(df, lp, split="dev", n=100, model_name="TowerInstruct-7B-w-chat-empty-sys", context_type="full_context_empty_sys", data_name="wmt24_chat"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')
  try:
    xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
    yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")
  except:
    xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
    yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")


  xx_yy_generations = np.array(xx_yy_generations).reshape((len(xx_yy_generations)//n, n))
  yy_xx_generations = np.array(yy_xx_generations).reshape((len(yy_xx_generations)//n, n))

  assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
  translations = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(xx_yy_generations[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(yy_xx_generations[k])
      k+=1
  return np.array(translations)


def read_file(fname, unescape_newline=True):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    if unescape_newline:
      output = [l.replace("\\n", "\n") for l in output]
    return output

def get_dataframe(split, lang_pair, data_name='wmt24_chat', model_name='TowerInstruct-7B-w-chat-empty-sys', context_type="full_context_empty_sys"):
    df = pd.read_csv(f"../tacl_results/{data_name}/{split}.{lang_pair}.csv",
                    keep_default_na=False,
        na_values=["NaN"],)
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    output_reshaped = get_candidates(df, lang_pair, split, data_name=data_name, model_name=model_name, context_type=context_type)

    return df, output_reshaped

In [5]:
split="test"
lang_pair = "en-de"
data_name="bcontrast"
# model_name = "TowerInstruct-7B-v0.2"
# inf_context="no_context"

# df, output_reshaped = get_dataframe(split, lang_pair, data_name, 
#                                     model_name, inf_context)
df, output_reshaped = get_dataframe(split, lang_pair, data_name)
n_sent, num_samples = output_reshaped.shape

In [6]:

def build_embeddings(sources, translations, comet_model, batch_size):

    src_batches = [
        sources[i : i + batch_size] for i in range(0, len(sources), batch_size)
    ]
    src_inputs = [comet_model.encoder.prepare_sample(batch) for batch in src_batches]
    mt_batches = [
        translations[i : i + batch_size]
        for i in range(0, len(translations), batch_size)
    ]
    mt_inputs = [comet_model.encoder.prepare_sample(batch) for batch in mt_batches]

    src_embeddings = []
    with torch.no_grad():
        for batch in src_inputs:
            input_ids = batch["input_ids"].to(comet_model.device)
            attention_mask = batch["attention_mask"].to(comet_model.device)
            src_embeddings.append(
                comet_model.get_sentence_embedding(input_ids, attention_mask)
            )
    src_embeddings = torch.vstack(src_embeddings)

    mt_embeddings = []
    with torch.no_grad():
        for batch in tqdm(mt_inputs, desc="Encoding sentences...", dynamic_ncols=True):
            input_ids = batch["input_ids"].to(comet_model.device)
            attention_mask = batch["attention_mask"].to(comet_model.device)
            mt_embeddings.append(
                comet_model.get_sentence_embedding(input_ids, attention_mask)
            )
    mt_embeddings = torch.vstack(mt_embeddings)

    return src_embeddings, mt_embeddings

In [7]:

def run_mbr(comet_metric, sources, translations, n_sent, num_samples, batch_size=64, use_context=False, device_id=0):
    
    if use_context:
        comet_metric.enable_context()
    else:
        comet_metric.use_context = False

    comet_metric.eval()
    comet_metric.to(device_id)

    src_embeddings, mt_embeddings = build_embeddings(
        sources, translations, comet_metric, batch_size
    )

    src_embeddings = src_embeddings.reshape(n_sent, num_samples, -1)

    mt_embeddings = mt_embeddings.reshape(n_sent, num_samples, -1)
    mbr_matrix = torch.zeros(n_sent, num_samples)

    with torch.no_grad():
        # Loop over all source sentences
        for i in tqdm(
            range(mbr_matrix.shape[0]), desc="MBR Scores...", dynamic_ncols=True
        ):
            source = src_embeddings[i, :]
            # Loop over all hypothesis
            for j in range(mbr_matrix.shape[1]):
                translation = mt_embeddings[i, j, :].repeat(num_samples, 1)
                # Score current hypothesis against all others
                pseudo_refs = mt_embeddings[i, :]
                scores = comet_metric.estimate(source, translation, pseudo_refs)["score"]
                scores = torch.cat([scores[0:j], scores[j + 1 :]])
                mbr_matrix[i, j] = scores.mean()
    return mbr_matrix.cpu().detach().numpy()


In [8]:
def add_context_across(orig_txt: List[str], context_same: List[str], context_other: List[str], 
                sender_ids: List[str], sep_token: str = "</s>", ws: int = 2) -> List[str]:
    if not (len(orig_txt) == len(context_same)== len(context_other)):
        raise Exception(f'Lengths should match: len(orig_txt)={len(orig_txt)}, len(context_same)={len(context_same)}, len(context_other)={len(context_other)}')
    context_all = []
    for i in range(len(orig_txt)):
      context_window = []
      for j in range(max(0, i - ws), i):
        if sender_ids[j] == sender_ids[i]:
            context_window.append(context_same[j])
        else:
            context_window.append(context_other[j])
      context_all.append(context_window)

    augm_txt = []
    for i in range(len(orig_txt)):
      context = context_all[i]
      augm_txt.append(" {} ".format(sep_token).join(context + [orig_txt[i]]))

    return augm_txt

In [9]:
def add_context_within_lp(orig_txt: List[str], context: List[str], doc_ids: List[str], sep_token: str = "</s>", ws: int = 2) -> List[str]:
    if not (len(orig_txt) == len(context) == len(doc_ids)):
        raise Exception(f'Lengths should match: len(orig_txt)={len(orig_txt)}, len(context)={len(context)}, len(doc_ids)={len(doc_ids)}')
    i, k = 0, 0
    augm_txt = []
    doc_id = doc_ids[0]
    while i < len(orig_txt):
        if doc_ids[i] == doc_id:
            context_window = context[i - min(k, ws):i]
            augm_txt.append(" {} ".format(sep_token).join(context_window + [orig_txt[i]]))
            i += 1
        else:
            doc_id = doc_ids[i]
            k = -1
        k += 1
    return augm_txt 

In [10]:
def get_source_and_translations(df, output_reshaped, use_context, context_size=2, context_source="source", context_mt="source", context_type="across"):
    if use_context:
        outputs_with_context = []
        sources_with_context = []
        for i in range(len(output_reshaped.T)):
            df["out"] = output_reshaped.T[i]
            if context_type=="across":
                outputs_curr = []
                src_curr = []
                for _, df_group in df.groupby(["doc_id"], sort=False):
                    src_with_context = add_context_across(orig_txt=df_group["source"].to_list(),
                                                    context_same=df_group[context_source].to_list(),
                                                    context_other=df_group[context_mt].to_list(),
                                                    sender_ids=df_group["sender"].to_list(),
                                                    sep_token=comet_metric.encoder.tokenizer.sep_token,
                                                    ws=context_size,)
                    out_with_context = add_context_across(orig_txt=df_group["out"].to_list(),
                                                    context_same=df_group[context_mt].to_list(),
                                                    context_other=df_group[context_source].to_list(),
                                                    sender_ids=df_group["sender"].to_list(),
                                                    sep_token=comet_metric.encoder.tokenizer.sep_token,
                                                    ws=context_size,)
                    outputs_curr.extend(out_with_context)
                    src_curr.extend(src_with_context)
            
            else:
                outputs_curr = []
                src_curr = []
                for _, df_group in df.groupby(["lp"], sort=False):
                    src_with_context = add_context_within_lp(orig_txt=df_group["source"].to_list(),
                                                        context=df_group[context_source].to_list(),
                                                        doc_ids=df_group["doc_id"].to_list(),
                                                        sep_token=comet_metric.encoder.tokenizer.sep_token,
                                                        ws=context_size,)
                    out_with_context = add_context_within_lp(orig_txt=df_group["out"].to_list(),
                                                        context=df_group[context_mt].to_list(),
                                                        doc_ids=df_group["doc_id"].to_list(),
                                                        sep_token=comet_metric.encoder.tokenizer.sep_token,
                                                        ws=context_size,)
                    outputs_curr.extend(out_with_context)
                    src_curr.extend(src_with_context)
            outputs_with_context.append(outputs_curr)
            sources_with_context.append(src_curr)
                
        sources = list(chain.from_iterable(np.array(sources_with_context).T))
        outputs = list(chain.from_iterable(np.array(outputs_with_context).T))
        
    else:
        outputs = list(chain.from_iterable(output_reshaped))
        sources = list(np.repeat(df["source"].to_list(), num_samples))
    
    return sources, outputs

In [11]:
sources, outputs = get_source_and_translations(df, output_reshaped, False)

In [12]:
len(sources), len(outputs)

(210000, 210000)

In [13]:
mbr_matrix = run_mbr(comet_metric, 
                         sources, 
                         outputs, 
                         n_sent,
                         num_samples,
                         batch_size=256)

col_argmax = np.argmax(mbr_matrix, axis=1)
df["comet-best"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]

MBR Scores...: 100%|██████████| 2100/2100 [04:05<00:00,  8.56it/s]


In [14]:
comet_metric.use_context = False

df[f"comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df["comet-best"], df["reference"])], 
                batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
df[f"chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df["comet-best"].to_list(), df["reference"].to_list())]

for lp, lp_df in df.groupby("lp"):
    print(lp, lp_df[f"comet"].mean(), lp_df[f"chrf"].mean(), sacrebleu.corpus_chrf(lp_df["comet-best"].to_list(), [lp_df["reference"].to_list()]))


/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [6]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Predicting DataLoader 0: 100%|██████████| 33/33 [00:08<00:00,  3.73it/s]


	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
de-en 0.9233751324734496 72.90864240015728 chrF2 = 69.71
en-de 0.9093988647738707 67.58154299599354 chrF2 = 67.84


In [15]:
use_context=True
sources, outputs = get_source_and_translations(df, output_reshaped, use_context, context_mt="comet-best")

In [16]:
len(sources), len(outputs)

(210000, 210000)

In [17]:
mbr_matrix = run_mbr(comet_metric, 
                         sources, 
                         outputs, 
                         n_sent,
                         num_samples,
                         batch_size=64,
                         use_context=use_context)

col_argmax = np.argmax(mbr_matrix, axis=1)
df["mbr-best"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]


MBR Scores...: 100%|██████████| 2100/2100 [04:07<00:00,  8.47it/s]


In [18]:
comet_metric.use_context = False

df[f"comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df["mbr-best"], df["reference"])], 
                batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
df[f"chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df["mbr-best"].to_list(), df["reference"].to_list())]

for lp, lp_df in df.groupby("lp"):
    print(lp, lp_df[f"comet"].mean(), lp_df[f"chrf"].mean(), sacrebleu.corpus_chrf(lp_df["mbr-best"].to_list(), [lp_df["reference"].to_list()]))


/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [6]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Predicting DataLoader 0: 100%|██████████| 33/33 [00:08<00:00,  3.73it/s]


de-en 0.9221492970002104 72.75475982762396 chrF2 = 69.62
en-de 0.9062641473965422 67.7811653080289 chrF2 = 67.94


In [19]:
use_context=True
sources, outputs = get_source_and_translations(df, output_reshaped, use_context)

In [20]:
mbr_matrix = run_mbr(comet_metric, 
                         sources, 
                         outputs, 
                         n_sent,
                         num_samples,
                         batch_size=64,
                         use_context=use_context)

col_argmax = np.argmax(mbr_matrix, axis=1)
df["mbr-best-src"]  = output_reshaped[np.arange(len(col_argmax)), col_argmax]


MBR Scores...: 100%|██████████| 2100/2100 [04:06<00:00,  8.53it/s]


In [21]:
comet_metric.use_context = False

df[f"comet"]  = comet_metric.predict([{"mt": y, "ref":z, "src": x} for x, y, z in zip(df["source"], df["mbr-best-src"], df["reference"])], 
                batch_size=64, gpus=1, progress_bar=True, devices=[0])['scores']
df[f"chrf"] = [sacrebleu.sentence_chrf(x, [y]).score for (x, y) in zip(df["mbr-best-src"].to_list(), df["reference"].to_list())]

for lp, lp_df in df.groupby("lp"):
    print(lp, lp_df[f"comet"].mean(), lp_df[f"chrf"].mean(), sacrebleu.corpus_chrf(lp_df["mbr-best-src"].to_list(), [lp_df["reference"].to_list()]))


/mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/sweta/envs/chat-qe-env/lib/python3.10/site ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [6]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Predicting DataLoader 0: 100%|██████████| 33/33 [00:08<00:00,  3.70it/s]


de-en 0.9224301624692584 72.83206705991526 chrF2 = 69.65
en-de 0.9058171979431967 67.35365531100776 chrF2 = 67.69
